In [0]:
# -*- coding: utf-8 -*-
# Notebook: /Workspace/Users/jorgee.lopez@adres.gov.co/mlops_canvas/notebooks/database

from pathlib import Path
from pyspark.sql import SparkSession
import shutil

# ───────────────── 0) Rutas locale  ─────────────────
PROJECT_ROOT = Path("/Workspace/Users/jorgee.lopez@adres.gov.co/mlops_canvas")
OUT_DIR      = PROJECT_ROOT / "data" / "raw" / "complete"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / "df.parquet"          # archivo final único
TMP_DIR  = OUT_DIR / "_tmp_parquet"         # carpeta temporal local

# ───────────────── 1) Spark ─────────────────
spark = SparkSession.builder.getOrCreate()
try:
    spark.sql("USE CATALOG mipres_catalog")
except Exception as e:
    print(f"[Aviso] No se pudo usar catálogo: {e}. Seguimos.")

BASE_TBL = "mipres_catalog.bronze_db_mipres_suministro.dbo_tsum_tx"

# ───────────────── 2) SQL ─────────────────
sql = f"""
WITH base_w AS (
  SELECT
    date_trunc('week', FechaRegistro) AS semana,
    FechaRegistro,
    TipoTecnologia,
    RegimenPrescripcion,
    CodigoETPrescripcion,
    CodigoEPSPrescripcion,
    struct(TipoIDPaciente, NoIDPaciente) AS id_paciente
  FROM {BASE_TBL}
),
marcas AS (
  SELECT
    b.*,
    MIN(semana) OVER (PARTITION BY id_paciente) AS primera_semana_global
  FROM base_w b
),
sem_core AS (
  SELECT
    semana,
    COUNT(*)                                        AS total_suministros_semana,
    COUNT(DISTINCT id_paciente)                     AS total_pacientes_unicos_semana,
    SUM(CASE WHEN TipoTecnologia='M' THEN 1 ELSE 0 END) AS n_tipo_m,
    SUM(CASE WHEN TipoTecnologia='P' THEN 1 ELSE 0 END) AS n_tipo_p,
    SUM(CASE WHEN TipoTecnologia='D' THEN 1 ELSE 0 END) AS n_tipo_d,
    SUM(CASE WHEN TipoTecnologia='N' THEN 1 ELSE 0 END) AS n_tipo_n,
    SUM(CASE WHEN TipoTecnologia='S' THEN 1 ELSE 0 END) AS n_tipo_s,
    SUM(CASE WHEN RegimenPrescripcion='C' THEN 1 ELSE 0 END) AS n_reg_c,
    SUM(CASE WHEN RegimenPrescripcion='S' THEN 1 ELSE 0 END) AS n_reg_s,
    SUM(CASE WHEN RegimenPrescripcion IS NULL OR RegimenPrescripcion NOT IN ('C','S') THEN 1 ELSE 0 END) AS n_reg_otro
  FROM base_w
  GROUP BY semana
),
hhi_w AS (
  SELECT
    semana,
    SUM( POW(depto_cnt / total_semana, 2) ) AS hhi_et_semana
  FROM (
    SELECT
      semana,
      CodigoETPrescripcion,
      COUNT(*) AS depto_cnt,
      SUM(COUNT(*)) OVER (PARTITION BY semana) AS total_semana
    FROM base_w
    GROUP BY semana, CodigoETPrescripcion
  ) t
  GROUP BY semana
),
shares_w AS (
  SELECT
    c.semana,
    c.total_suministros_semana,
    c.total_pacientes_unicos_semana,
    (n_tipo_m / NULLIF(c.total_suministros_semana,0)) AS share_tipo_medicamento,
    (n_tipo_p / NULLIF(c.total_suministros_semana,0)) AS share_tipo_procedimiento,
    (n_tipo_d / NULLIF(c.total_suministros_semana,0)) AS share_tipo_dispositivo,
    (n_tipo_n / NULLIF(c.total_suministros_semana,0)) AS share_tipo_nutricional,
    (n_tipo_s / NULLIF(c.total_suministros_semana,0)) AS share_tipo_servicio_comp,
    (n_reg_c / NULLIF(c.total_suministros_semana,0))  AS share_regimen_contributivo,
    (n_reg_s / NULLIF(c.total_suministros_semana,0))  AS share_regimen_subsidiado,
    (n_reg_otro / NULLIF(c.total_suministros_semana,0)) AS share_regimen_otro
  FROM sem_core c
),
nuevos_pref AS (
  SELECT
    semana,
    COUNT(DISTINCT CASE WHEN primera_semana_global=semana AND CodigoEPSPrescripcion LIKE 'EPS%%' THEN id_paciente END) AS u_nuevos_pref_eps_semana,
    COUNT(DISTINCT CASE WHEN primera_semana_global=semana AND CodigoEPSPrescripcion LIKE 'EAS%%' THEN id_paciente END) AS u_nuevos_pref_eas_semana,
    COUNT(DISTINCT CASE WHEN primera_semana_global=semana AND CodigoEPSPrescripcion LIKE 'ESS%%' THEN id_paciente END) AS u_nuevos_pref_ess_semana,
    COUNT(DISTINCT CASE WHEN primera_semana_global=semana AND CodigoEPSPrescripcion LIKE 'CCF%%' THEN id_paciente END) AS u_nuevos_pref_ccf_semana
  FROM marcas
  GROUP BY semana
),
y_w AS (
  SELECT
    semana,
    COUNT(DISTINCT CASE WHEN primera_semana_global=semana THEN id_paciente END) AS y_usuarios_nuevos_semana
  FROM marcas
  GROUP BY semana
),
ensamblado AS (
  SELECT
    s.semana,
    y.y_usuarios_nuevos_semana,
    s.total_suministros_semana,
    s.total_pacientes_unicos_semana,
    s.share_tipo_medicamento,
    s.share_tipo_procedimiento,
    s.share_tipo_dispositivo,
    s.share_tipo_nutricional,
    s.share_tipo_servicio_comp,
    s.share_regimen_contributivo,
    s.share_regimen_subsidiado,
    s.share_regimen_otro,
    h.hhi_et_semana,
    p.u_nuevos_pref_eps_semana,
    p.u_nuevos_pref_eas_semana,
    p.u_nuevos_pref_ess_semana,
    p.u_nuevos_pref_ccf_semana
  FROM shares_w s
  LEFT JOIN hhi_w  h USING (semana)
  LEFT JOIN nuevos_pref p USING (semana)
  LEFT JOIN y_w y     USING (semana)
),
lags_t4_only AS (
  SELECT
    semana,
    y_usuarios_nuevos_semana,
    LAG(total_suministros_semana,      4) OVER (ORDER BY semana) AS total_suministros_t4,
    LAG(total_pacientes_unicos_semana, 4) OVER (ORDER BY semana) AS total_pacientes_unicos_t4,
    LAG(share_tipo_medicamento,        4) OVER (ORDER BY semana) AS share_tipo_medicamento_t4,
    LAG(share_tipo_procedimiento,      4) OVER (ORDER BY semana) AS share_tipo_procedimiento_t4,
    LAG(share_tipo_dispositivo,        4) OVER (ORDER BY semana) AS share_tipo_dispositivo_t4,
    LAG(share_tipo_nutricional,        4) OVER (ORDER BY semana) AS share_tipo_nutricional_t4,
    LAG(share_tipo_servicio_comp,      4) OVER (ORDER BY semana) AS share_tipo_servicio_comp_t4,
    LAG(share_regimen_contributivo,    4) OVER (ORDER BY semana) AS share_regimen_contributivo_t4,
    LAG(share_regimen_subsidiado,      4) OVER (ORDER BY semana) AS share_regimen_subsidiado_t4,
    LAG(share_regimen_otro,            4) OVER (ORDER BY semana) AS share_regimen_otro_t4,
    LAG(hhi_et_semana,                 4) OVER (ORDER BY semana) AS hhi_et_t4,
    LAG(u_nuevos_pref_eps_semana,      4) OVER (ORDER BY semana) AS u_nuevos_pref_eps_t4,
    LAG(u_nuevos_pref_eas_semana,      4) OVER (ORDER BY semana) AS u_nuevos_pref_eas_t4,
    LAG(u_nuevos_pref_ess_semana,      4) OVER (ORDER BY semana) AS u_nuevos_pref_ess_t4,
    LAG(u_nuevos_pref_ccf_semana,      4) OVER (ORDER BY semana) AS u_nuevos_pref_ccf_t4
  FROM ensamblado
),
final_ready AS (
  SELECT *
  FROM lags_t4_only
  WHERE total_suministros_t4 IS NOT NULL
)
SELECT *
FROM final_ready
ORDER BY semana
"""
df = spark.sql(sql)

# ───────────────── 3) Vista rápida (opcional) ─────────────────
try:
    display(df)
except Exception:
    df.show(20, truncate=False)

# ───────────────── 4) Exportar a archivo único Parquet en /Workspace ─────────────────
# Spark escribe en carpeta: usamos 'file:' para el filesystem local del driver,
# y luego movemos el único part-*.parquet a df.parquet.

# 4.1 limpiar tmp y escribir 1 solo part-file
if TMP_DIR.exists():
    shutil.rmtree(TMP_DIR)
# IMPORTANTE: usar esquema 'file:' para que Spark escriba en FS local (NO dbfs)
spark.write = df.coalesce(1).write.mode("overwrite")
spark.write.parquet(f"file:{TMP_DIR.as_posix()}")

# 4.2 localizar part-*.parquet y renombrar
parts = list(TMP_DIR.glob("part-*.parquet"))
if not parts:
    # ayuda de depuración
    print("[Depuración] Contenido de TMP_DIR:", TMP_DIR)
    for p in TMP_DIR.iterdir():
        print(" -", p)
    raise RuntimeError("No se encontró part-*.parquet en el directorio temporal local.")
part_file = parts[0]

# 4.3 reemplazar si existía un df.parquet previo
if OUT_FILE.exists():
    OUT_FILE.unlink()

# mover part -> df.parquet
part_file.replace(OUT_FILE)

# 4.4 limpiar temporal
shutil.rmtree(TMP_DIR, ignore_errors=True)

print(f"✅ Archivo Parquet único escrito en: {OUT_FILE}")
